In [ ]:
#%run prelude.rc

import enum
import importlib.util
import sys
from pathlib import Path

import pyarrow

import polars as pl
import numpy as np
import scipy.integrate as integrate
import HErmes as he
import HErmes.fitting as fit
import scipy.stats as st
import matplotlib

from scipy.spatial.transform import Rotation as rot
from datetime import datetime, UTC, timezone
from glob import glob

#pybindings
from pathlib import Path
import dashi as d
d.visual()
import tqdm


import matplotlib.pyplot as plt
import charmingbeauty as cb
lo = cb.layout
cb.visual.set_style_present()


import re
!export DJANGO_ALLOW_ASYNC_UNSAFE=1
import os
from matplotlib import font_manager
from matplotlib import rcParams


os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'
plt.rcParams.update({'text.usetex' : False})


from matplotlib import font_manager


rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Open Sans']



import numpy as np
import polars as pl
import numpy as np
import polars as pl

def average_every_n_by_board(df, n, time_col="timestamp", board_col="board_id"):
    if len(df) == 0:
        return df

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe')
    if board_col not in df.columns:
        raise ValueError(f'{board_col} not in dataframe')

    out = []
    boards = np.unique(df[board_col].to_numpy())
       
    for b in boards:
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )
        
        m = len(sub)
        if m == 0:
            continue

        # make consecutive bins AFTER sorting
        bin_id = np.arange(m) // n
        sub = sub.with_columns(pl.Series("bin_id", bin_id))
        
        exprs = []
        for c, dt in zip(sub.columns, sub.dtypes):
            if c in [board_col, "bin_id"]:
                continue
            if c == time_col:
                exprs.append(pl.col(c).mean().alias(c))
            elif dt.is_numeric():
                exprs.append(pl.col(c).mean().alias(c))
                
        agg = (
            sub.group_by("bin_id", maintain_order=True)
               .agg(exprs)
               .with_columns(pl.lit(b).alias(board_col))
               .drop("bin_id")
               .sort(time_col)
        )
        
        out.append(agg)

    if not out:
        return pl.DataFrame()

    return pl.concat(out).sort([board_col, time_col])





def average_every_n(df, n, time_col="timestamp"):
    if len(df) == 0:
        return df

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe')

    sub = df.sort(time_col)

    m = len(sub)
    bin_id = np.arange(m) // n
    sub = sub.with_columns(pl.Series("bin_id", bin_id))

    exprs = []
    for c, dt in zip(sub.columns, sub.dtypes):
        if c == "bin_id":
            continue
        if c == time_col:
            exprs.append(pl.col(c).mean().alias(c))
        elif dt.is_numeric():
            exprs.append(pl.col(c).mean().alias(c))

    return (
        sub.group_by("bin_id", maintain_order=True)
           .agg(exprs)
           .drop("bin_id")
           .sort(time_col)
    )

In [ ]:

%matplotlib inline


plt.rcParams["font.family"] = "DejaVu Sans"
import gondola as gon
import time

files = gon.io.grace_get_telemetry_binaries(
    1766939800, #start 1765835400
    1766949800,
    #1767039800, #end time 1767979800 (testing it is for the random 100,000 seconds of flight)
    '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
)


lpt = None
toml_find = False


import polars as pl
import gondola as gon



# fresh in-memory monitors
pa   = gon.monitoring.PAMoniDataSeries()
cpuM = gon.monitoring.CPUMoniDataSeries()
rbM  = gon.monitoring.RBMoniDataSeries()
ltbM = gon.monitoring.LTBMoniDataSeries()
mtbM = gon.monitoring.MasterTriggerHBSeries()

fileCount = 0

for f in files:
    sf = str(f)

    pa.add_telemetryfile(sf)
    cpuM.add_telemetryfile(sf)
    rbM.add_telemetryfile(sf)
    ltbM.add_telemetryfile(sf)
    mtbM.add_telemetryfile(sf)

    fileCount += 1
dfPA = pa.get_dataframe()
dfCPU = cpuM.get_dataframe()
dfRB = rbM.get_dataframe()
dfLTB = ltbM.get_dataframe()
dfMTB = mtbM.get_dataframe()


print("n files =", len(files))
print("len(dfPA)  =", len(dfPA))
print("len(dfCPU) =", len(dfCPU))
print("len(dfRB)  =", len(dfRB))
print("len(dfLTB) =", len(dfLTB))
print("len(dfMTB) =", len(dfMTB))

# consistant throughout flight ...

rb_moni_interval = 10.0# was 20
pb_moni_every_x = 2.0
pa_moni_every_x = 2.0
ltb_moni_every_x = 2.0
mtb_moni_interval = 10


def print_cols(name, df):
    print(f"\n{name} columns ({len(df.columns)}):")
    for c in df.columns:
        print("  ", c)



In [ ]:



print_cols("PA", dfPA)
print_cols("CPU", dfCPU)
print_cols("RB", dfRB)
print_cols("LTB", dfLTB)
print_cols("MTB", dfMTB)



In [ ]:
pa_temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
pa_bias_cols = [c for c in dfPA.columns if c.startswith("biases")]

rb_temp_cols = [c for c in dfRB.columns if c.startswith("tmp_")]
rb_voltage_cols = [c for c in dfRB.columns if c.endswith("_voltage")]
rb_current_cols = [c for c in dfRB.columns if c.endswith("_current")]
rb_power_cols = [c for c in dfRB.columns if c.endswith("_power")]
rb_env_cols = ["pressure", "humidity"]

cpu_temp_cols = [c for c in dfCPU.columns if "temp" in c.lower()]
cpu_freq_cols = [c for c in dfCPU.columns if "freq" in c.lower()]

ltb_reasonable_cols = ["trenz_temp", "ltb_temp", "thresh0", "thresh1", "thresh2"]

mtb_rate_cols = [
    "trate", "lost_trate", "rb_lost_rate", "tiu_busy_rate",
    "trg_lost_trg_rate", "gaps_blocked_rate", "track_blocked_rate",
    "any_blocked_rate", "trkctrl_blocked_rate"
]


def finite_mask(*arrays):
    mask = np.ones(len(arrays[0]), dtype=bool)
    for a in arrays:
        a = np.asarray(a)
        mask &= np.isfinite(a)
    return mask

def range_mask(x, xmin=None, xmax=None):
    x = np.asarray(x)
    mask = np.isfinite(x)
    if xmin is not None:
        mask &= x >= xmin
    if xmax is not None:
        mask &= x <= xmax
    return mask


# PA
PA_TEMP_MIN, PA_TEMP_MAX = -50, 60
PA_BIAS_MIN, PA_BIAS_MAX = 45, 65

# RB
RB_TEMP_MIN, RB_TEMP_MAX = -50, 90
RB_VOLT_MIN, RB_VOLT_MAX = -5, 10
RB_CURR_MIN, RB_CURR_MAX = -1, 10
RB_PWR_MIN,  RB_PWR_MAX  = -1, 50
HUM_MIN, HUM_MAX = 0, 100
PRESS_MIN, PRESS_MAX = 0, 1200

# CPU
CPU_TEMP_MIN, CPU_TEMP_MAX = -20, 120
CPU_FREQ_MIN, CPU_FREQ_MAX = 0, 5000

# LTB
LTB_TEMP_MIN, LTB_TEMP_MAX = -50, 90
THR_MIN, THR_MAX = 0, 1000

# MTB
RATE_MIN, RATE_MAX = 0, 1e6





In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 13))

# -------------------------
# 1) PA: temp vs bias
# -------------------------
ax = axes[0, 0]
for t, b in zip(pa_temp_cols, pa_bias_cols):
    x = dfPA[t].to_numpy()
    y = dfPA[b].to_numpy()
    m = finite_mask(x, y) & range_mask(x, PA_TEMP_MIN, PA_TEMP_MAX) & range_mask(y, PA_BIAS_MIN, PA_BIAS_MAX)
    ax.scatter(x[m], y[m], s=8, alpha=0.35)
ax.set_xlabel("PA temperature (C)")
ax.set_ylabel("PA bias (V)")
ax.set_title("PA bias vs temperature")

# -------------------------
# 2) PA: temp consistency
# -------------------------
ax = axes[0, 1]
for i in range(len(pa_temp_cols)-1):
    x = dfPA[pa_temp_cols[i]].to_numpy()
    y = dfPA[pa_temp_cols[i+1]].to_numpy()
    m = finite_mask(x, y) & range_mask(x, PA_TEMP_MIN, PA_TEMP_MAX) & range_mask(y, PA_TEMP_MIN, PA_TEMP_MAX)
    ax.scatter(x[m], y[m], s=8, alpha=0.35)
ax.set_xlabel("temp_i")
ax.set_ylabel("temp_(i+1)")
ax.set_title("PA temperature consistency")

# -------------------------
# 3) PA: bias spread per board row
# -------------------------
ax = axes[0, 2]
bias_matrix = np.column_stack([dfPA[c].to_numpy() for c in pa_bias_cols])
bias_ok = np.all(np.isfinite(bias_matrix), axis=1)
bias_ok &= np.all((bias_matrix > PA_BIAS_MIN) & (bias_matrix < PA_BIAS_MAX), axis=1)
spread = np.std(bias_matrix[bias_ok], axis=1)
ax.hist(spread, bins=30)
ax.set_xlabel("row-wise sigma(bias)")
ax.set_ylabel("count")
ax.set_title("PA bias spread")

# -------------------------
# 4) RB: board temp vs rate
# -------------------------
ax = axes[1, 0]
x = dfRB["tmp_drs"].to_numpy()
y = dfRB["rate"].to_numpy()
m = finite_mask(x, y) & range_mask(x, RB_TEMP_MIN, RB_TEMP_MAX) & range_mask(y, RATE_MIN, RATE_MAX)
ax.scatter(x[m], y[m], s=10, alpha=0.5)
ax.set_xlabel("RB tmp_drs (C)")
ax.set_ylabel("RB rate")
ax.set_title("RB rate vs DRS temp")

# -------------------------
# 5) RB: humidity vs temperature
# -------------------------
ax = axes[1, 1]
x = dfRB["humidity"].to_numpy()
y = dfRB["tmp_bm280"].to_numpy()
m = finite_mask(x, y) & range_mask(x, HUM_MIN, HUM_MAX) & range_mask(y, RB_TEMP_MIN, RB_TEMP_MAX)
ax.scatter(x[m], y[m], s=10, alpha=0.5)
ax.set_xlabel("humidity (%)")
ax.set_ylabel("RB tmp_bm280 (C)")
ax.set_title("RB humidity vs BM280 temp")

# -------------------------
# 6) RB: voltage sanity
# -------------------------
ax = axes[1, 2]
for c in ["p3v3_voltage", "p3v5_voltage", "zynq_voltage", "drs_dvdd_voltage", "adc_dvdd_voltage"]:
    if c in dfRB.columns:
        y = dfRB[c].to_numpy()
        x = np.arange(len(y))
        m = finite_mask(x, y) & range_mask(y, RB_VOLT_MIN, RB_VOLT_MAX)
        ax.plot(x[m], y[m], ".", markersize=2, label=c)
ax.set_xlabel("row")
ax.set_ylabel("voltage (V)")
ax.set_title("RB rail voltages")
ax.legend(fontsize=8)

# -------------------------
# 7) LTB: thresholds
# -------------------------
ax = axes[2, 0]
for c in ["thresh0", "thresh1", "thresh2"]:
    y = dfLTB[c].to_numpy()
    x = np.arange(len(y))
    m = finite_mask(x, y) & range_mask(y, THR_MIN, THR_MAX)
    ax.plot(x[m], y[m], ".", markersize=3, label=c)
ax.set_xlabel("row")
ax.set_ylabel("threshold")
ax.set_title("LTB thresholds")
ax.legend(fontsize=8)

# -------------------------
# 8) MTB: trigger losses
# -------------------------
ax = axes[2, 1]
for c in ["trate", "lost_trate", "rb_lost_rate", "any_blocked_rate"]:
    if c in dfMTB.columns:
        y = dfMTB[c].to_numpy()
        x = np.arange(len(y))
        m = finite_mask(x, y) & range_mask(y, RATE_MIN, RATE_MAX)
        ax.plot(x[m], y[m], ".", markersize=3, label=c)
ax.set_xlabel("row")
ax.set_ylabel("rate")
ax.set_title("MTB rates / losses")
ax.legend(fontsize=8)

# -------------------------
# 9) CPU temps
# -------------------------
ax = axes[2, 2]
for c in cpu_temp_cols:
    y = dfCPU[c].to_numpy()
    x = np.arange(len(y))
    m = finite_mask(x, y) & range_mask(y, CPU_TEMP_MIN, CPU_TEMP_MAX)
    ax.plot(x[m], y[m], ".", markersize=3, label=c)
ax.set_xlabel("row")
ax.set_ylabel("temperature (C)")
ax.set_title("CPU temperatures")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
def corr_subset(df, cols, cuts=None):
    arrs = []
    names = []
    n = len(df)
    mask = np.ones(n, dtype=bool)

    for c in cols:
        x = df[c].to_numpy()
        mask &= np.isfinite(x)
        if cuts and c in cuts:
            lo, hi = cuts[c]
            if lo is not None:
                mask &= x >= lo
            if hi is not None:
                mask &= x <= hi

    for c in cols:
        arrs.append(df[c].to_numpy()[mask])
        names.append(c)

    A = np.column_stack(arrs)
    C = np.corrcoef(A, rowvar=False)
    return names, C

def plot_corr(ax, names, C, title):
    im = ax.imshow(C, vmin=-1, vmax=1)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_title(title)
    return im
    

In [ ]:
def nearest_match(x_ref, x_other):
    x_ref = np.asarray(x_ref)
    x_other = np.asarray(x_other)

    if len(x_other) == 0:
        raise ValueError("nearest_match: reference comparison array is empty")

    if len(x_other) == 1:
        return np.zeros(len(x_ref), dtype=int)

    idx = np.searchsorted(x_other, x_ref)
    idx = np.clip(idx, 1, len(x_other) - 1)

    left = idx - 1
    right = idx

    choose_right = np.abs(x_other[right] - x_ref) < np.abs(x_other[left] - x_ref)
    return np.where(choose_right, right, left)



def getTempAv(dfPA):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]

    # stack into matrix: shape (n_rows, 16)
    temps = np.column_stack([dfPA[c].to_numpy() for c in temp_cols])

    # ignore NaNs automatically
    return np.nanmean(temps, axis=1)

    
def getTempAvB(dfPA, boardID):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]

    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .group_by("timestamp")
        .agg([
            *[pl.col(c).mean().alias(c) for c in temp_cols]
        ])
        .sort("timestamp")
    )

    if len(sub) == 0:
        return np.array([]), np.array([])

    temps = np.column_stack([sub[c].to_numpy() for c in temp_cols])
    t = sub["timestamp"].to_numpy()
    avg = np.nanmean(temps, axis=1)

    return t, avg

def match_by_board(dfPA, dfRB):
    pa_board = dfPA["board_id"].to_numpy()
    rb_board = dfRB["board_id"].to_numpy()

    t_pa = dfPA["timestamp"].to_numpy()
    t_rb = dfRB["timestamp"].to_numpy()

    idx_out = np.full(len(dfPA), -1, dtype=int)

    # loop over unique boards present in PA
    for b in np.unique(pa_board):

        pa_mask = (pa_board == b)
        rb_mask = (rb_board == b)

        if np.sum(rb_mask) == 0:
            continue  # no matching RB for this board

        t_pa_sub = t_pa[pa_mask]
        t_rb_sub = t_rb[rb_mask]

        idx_sub = nearest_match(t_pa_sub, t_rb_sub)

        # map back to full indices
        rb_indices = np.where(rb_mask)[0]
        idx_out[pa_mask] = rb_indices[idx_sub]
    return idx_out

    

In [ ]:
print(dfPA.columns)

In [ ]:
boardID = 7
t, temp_avg = getTempAvB(dfPA, boardID)

plt.figure()
plt.plot(t - t[0], temp_avg, ".")
plt.title(f"Average PA temperature, board {boardID}")
plt.xlabel("Time since start (s)")
plt.ylabel("Temp (C)")
plt.show()

In [ ]:


temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
boards = np.unique(dfPA["board_id"].to_numpy())

for b in boards:
    mask = (dfPA["board_id"].to_numpy() == b)

    if np.sum(mask) < 5:
        continue

    t = dfPA["timestamp"].to_numpy()[mask]

    temps = np.column_stack([
        dfPA[c].to_numpy()[mask] for c in temp_cols
    ])

    avg = np.nanmean(temps, axis=1)
    std = np.nanstd(temps, axis=1)

    plt.figure(figsize=(8,5))

    # individual channels
    for i in range(temps.shape[1]):
        plt.scatter(t, temps[:, i], alpha=0.3)

    # average
    plt.plot(t, avg, linewidth=2, label="avg")

    # spread band
    plt.fill_between(t, avg-std, avg+std, alpha=0.2)

    plt.xlabel("Time")
    plt.ylabel("Temp (C)")
    plt.title(f"Board {b} temps (with avg + spread)")
    plt.legend()

    plt.show()

In [ ]:
temp_avg = getTempAv(dfPA)

idx = match_by_board(dfPA, dfRB)
valid = idx >= 0

pa_board = dfPA["board_id"].to_numpy()
t_pa = dfPA["timestamp"].to_numpy()
t_rb = dfRB["timestamp"].to_numpy()
rb_rate = dfRB["rate"].to_numpy()



import matplotlib.pyplot as plt
import numpy as np

for b in np.unique(pa_board):

    mask = (pa_board == b) & valid

    if np.sum(mask) < 20:
        continue

    t = t_pa[mask]
    temp = temp_avg[mask]
    rate = rb_rate[idx[mask]]

    fig, axes = plt.subplots(2, 1, figsize=(8,6), sharex=True)

    # -----------------------
    # Temperature vs time
    # -----------------------
    axes[0].plot(t, temp, ".", markersize=3)
    axes[0].set_ylabel("Temp (C)")
    axes[0].set_title(f"Board {int(b)}")

    # -----------------------
    # Rate vs time
    # -----------------------
    axes[1].plot(t, rate, ".", markersize=3)
    axes[1].set_ylabel("Rate")
    axes[1].set_xlabel("Time")

    plt.tight_layout()
    plt.show()
    

In [ ]:
# bad slop

import polars as pl
import numpy as np
import gondola as gon
import polars as pl
import numpy as np
import gondola as gon


def compress_chunk(df, dt=10.0, time_col="timestamp", board_col=None):
    if len(df) == 0:
        return None

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe columns: {df.columns}')

    # assign absolute-time bins
    out = df.with_columns(
        (pl.col(time_col) / dt).floor().cast(pl.Int64).alias("tbin")
    )

    key_cols = ([board_col] if board_col is not None else []) + ["tbin"]

    exprs = [
        pl.len().alias("_count"),
        pl.col(time_col).sum().alias("_ts_weighted"),
    ]

    # sum all other numeric cols
    for c, dtp in zip(df.columns, df.dtypes):
        if c in key_cols or c == time_col:
            continue
        if dtp.is_numeric():
            exprs.append(pl.col(c).sum().alias(c))

    out = out.group_by(key_cols).agg(exprs)

    # force stable column order
    value_cols = [c for c in out.columns if c not in key_cols + ["_count", "_ts_weighted"]]
    out = out.select(key_cols + ["_count", "_ts_weighted"] + value_cols)

    return out


def merge_compressed(accum, chunk, time_col="timestamp", board_col=None):
    if chunk is None or len(chunk) == 0:
        return accum
    if accum is None:
        return chunk

    key_cols = ([board_col] if board_col is not None else []) + ["tbin"]

    # reorder chunk to match accum exactly
    chunk = chunk.select(accum.columns)

    merged = pl.concat([accum, chunk], how="vertical")

    exprs = [
        pl.col("_count").sum().alias("_count"),
        pl.col("_ts_weighted").sum().alias("_ts_weighted"),
    ]

    for c, dtp in zip(merged.columns, merged.dtypes):
        if c in key_cols or c in ["_count", "_ts_weighted"]:
            continue
        if dtp.is_numeric():
            exprs.append(pl.col(c).sum().alias(c))

    merged = merged.group_by(key_cols).agg(exprs)

    # stable order again
    value_cols = [c for c in merged.columns if c not in key_cols + ["_count", "_ts_weighted"]]
    merged = merged.select(key_cols + ["_count", "_ts_weighted"] + value_cols)

    return merged.sort(key_cols)


def finalize_compressed(df, time_col="timestamp", board_col=None):
    if df is None or len(df) == 0:
        return df

    key_cols = ([board_col] if board_col is not None else []) + ["tbin"]

    out = df.with_columns(
        (pl.col("_ts_weighted") / pl.col("_count")).alias(time_col)
    )

    non_avg_cols = set(key_cols + ["_count", "_ts_weighted", time_col])

    for c, dtp in zip(out.columns, out.dtypes):
        if c in non_avg_cols:
            continue
        if dtp.is_numeric():
            out = out.with_columns((pl.col(c) / pl.col("_count")).alias(c))

    # drop helper cols
    out = out.drop(["_count", "_ts_weighted"])

    sort_cols = ([board_col] if board_col is not None else []) + [time_col]
    return out.sort(sort_cols)

dfPA_accum  = None
dfRB_accum  = None
dfLTB_accum = None

chunk_size = 50

for i in range(0, len(files), chunk_size):
    chunk = files[i:i+chunk_size]

    pa   = gon.monitoring.PAMoniDataSeries()
    rbM  = gon.monitoring.RBMoniDataSeries()
    ltbM = gon.monitoring.LTBMoniDataSeries()

    for f in chunk:
        sf = str(f)
        pa.add_telemetryfile(sf)
        rbM.add_telemetryfile(sf)
        ltbM.add_telemetryfile(sf)

    dfPA_chunk  = pa.get_dataframe()
    dfRB_chunk  = rbM.get_dataframe()
    dfLTB_chunk = ltbM.get_dataframe()

    dfPA_comp  = compress_chunk(dfPA_chunk,  dt=10.0, time_col="timestamp", board_col="board_id")
    dfRB_comp  = compress_chunk(dfRB_chunk,  dt=10.0, time_col="timestamp", board_col="board_id")
    dfLTB_comp = compress_chunk(dfLTB_chunk, dt=10.0, time_col="timestamp", board_col="board_id")

    dfPA_accum  = merge_compressed(dfPA_accum,  dfPA_comp,  time_col="timestamp", board_col="board_id")
    dfRB_accum  = merge_compressed(dfRB_accum,  dfRB_comp,  time_col="timestamp", board_col="board_id")
    dfLTB_accum = merge_compressed(dfLTB_accum, dfLTB_comp, time_col="timestamp", board_col="board_id")

dfPA  = finalize_compressed(dfPA_accum,  time_col="timestamp", board_col="board_id")
dfRB  = finalize_compressed(dfRB_accum,  time_col="timestamp", board_col="board_id")
dfLTB = finalize_compressed(dfLTB_accum, time_col="timestamp", board_col="board_id") 


